# Extracting Conflicts of Interest from Scientific Papers with Local LLMs

![](assets/paper.png)

## Get the Authors

In [1]:
authors = "Claus F Vogelmeier, Konstantinos Kostikas, Juanzhi Fang, Hengfeng Tian, Bethan Jones, Christopher Ll Morgan, Robert Fogel, Florian S Gutzwiller, Hui Cao"

![](assets/author.png)

## Get Conflict of Interest

In [2]:
coi_statement = "CFV reports research support from AstraZeneca, Boehringer Ingelheim, GlaxoSmithKline, Grifols, Novartis, Bayer Schering, MSD, Pfizer, consultancy from AstraZeneca, Boehringer Ingelheim, CSL Behring, GlaxoSmithKline, Grifols, Menarini, Novartis, Teva, Cipla, and honoraria from AstraZeneca, Boehringer Ingelheim, CSL Behring, GlaxoSmithKline, Grifols, Menarini, Mundipharma, Novartis, Teva, Cipla. KK was an employee of Novartis Pharma AG at the time of the conduct of this study. JF is an employee of Novartis Pharmaceuticals Corporation owning stocks through the employment. HT is employee of KMK Consulting Inc. and providing consulting to Novartis, the sponsor of the study. BJ is an employee of Pharmatelligence who received funding from Novartis to conduct analyses for this study. CM is a consultant of Pharmatelligence who received funding from Novartis to conduct analyses for this study. RF is an employee of Novartis Pharmaceuticals Corporation owning stocks through the employment. FSG is an employee of Novartis Pharma AG owning stocks through the employment. HC is an employee of Novartis Pharmaceuticals Corporation owning stocks through the employment."

![](assets/coi.png)

## Create a prompt

In [3]:
prompt = f"""Take this conflict of interest statement and extract out a table with 
three columns (1) name of author (2) company (3) reasons for conflict. I will provide 
the list of authors. Make each row a conflict (relationship between an author and a company
for a particular reaason). An author may have multiple conflicts with the same company for 
different reasons, in that case make each row a separate conflict.

list of authors: 
{authors}

conflict of interest statement: 
{coi_statement}"""


## Run through a local model with structured outputs

This cell defines Pydantic models that constrain the model to return structured data as an array of author-company-conflict objects.

In [4]:
from pydantic import BaseModel

class Conflict(BaseModel):
    author: str
    company: str
    conflict: str

class ConflictsOfInterest(BaseModel):
    conflicts: list[Conflict]

This cell sends the prompt to the local model with the Pydantic schema enforced, then accesses the parsed response directly as a typed object and converts it to a DataFrame.

In [5]:
import pandas as pd
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
model = "meta-llama-3.1-8b-instruct"

response = client.beta.chat.completions.parse(
    model=model,
    messages=[{"role": "user", "content": prompt}],
    response_format=ConflictsOfInterest,
    temperature=0,
)

data = response.choices[0].message.parsed
df = pd.DataFrame([c.model_dump() for c in data.conflicts])
df

,author,company,conflict
0,Claus F Vogelmeier,AstraZeneca,research support
1,Claus F Vogelmeier,AstraZeneca,consultancy
2,Claus F Vogelmeier,AstraZeneca,honoraria
3,Claus F Vogelmeier,Boehringer Ingelheim,research support
4,Claus F Vogelmeier,Boehringer Ingelheim,consultancy
5,Claus F Vogelmeier,Boehringer Ingelheim,honoraria
6,Claus F Vogelmeier,Grifols,research support
7,Claus F Vogelmeier,Grifols,consultancy
8,Claus F Vogelmeier,Grifols,honoraria
9,Claus F Vogelmeier,GlaxoSmithKline,research support
